In [ ]:
%pip install -q -U langchain langchain-openai

In [ ]:
import os
from getpass import getpass
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

# Ask for the API key only if it is not already set
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

OpenAI API key: ··········


In [ ]:
# Car and bike data (approximate specs): (name, hp, top speed km/h)
CARS = [
    ("Bugatti Chiron Super Sport", 1578, 440),
    ("Koenigsegg Jesko",           1600, 480),
    ("Ferrari SF90 Stradale",       986, 340),
    ("Lamborghini Revuelto",       1015, 350),
    ("Pagani Huayra",               730, 370),
    ("McLaren 720S",                710, 341),
    ("Porsche 911 Turbo S",         640, 330),
]
BIKES = [
    ("Kawasaki Ninja H2R",   326, 400),
    ("Kawasaki Ninja H2",    228, 300),
    ("Ducati Panigale V4 R", 240, 299),
    ("Aprilia RSV4 Factory", 217, 300),
    ("BMW S 1000 RR",        205, 299),
    ("Yamaha YZF-R1M",       200, 299),
    ("Suzuki Hayabusa",      190, 299),
]

def best_match(data, name_hint, power_hp, top_speed_kmh):
    """Score each item by how close it is to the clues and return the best one."""
    best_score, best_item = -1, None
    for name, hp, speed in data:
        score = 0
        # Name clue: a partial name is enough (case does not matter)
        if name_hint and name_hint.lower() in name.lower():
            score += 2
        if power_hp:
            score += max(0, 1 - abs(hp - power_hp) / power_hp)          # closer value = higher score
        if top_speed_kmh:
            score += max(0, 1 - abs(speed - top_speed_kmh) / top_speed_kmh)
        if score > best_score:
            best_score, best_item = score, (name, hp, speed)
    name, hp, speed = best_item
    return f"{name} ({hp} hp, {speed} km/h), match score {best_score:.2f}"

# Tool 1: only for cars
@tool
def find_car(name: str = "", power_hp: float = 0, top_speed_kmh: float = 0) -> str:
    """Find the supercar that best matches the clues (name, power, top speed).
    Use only when the user asks about a car."""
    return best_match(CARS, name, power_hp, top_speed_kmh)

# Tool 2: only for bikes (same pattern as find_car)
@tool
def find_bike(name: str = "", power_hp: float = 0, top_speed_kmh: float = 0) -> str:
    """Find the superbike that best matches the clues (name, power, top speed).
    Use only when the user asks about a bike or motorcycle."""
    return best_match(BIKES, name, power_hp, top_speed_kmh)

# Tool 3: show the available names
@tool
def list_names(vehicle_type: str) -> str:
    """List the available vehicle names. vehicle_type must be 'car' or 'bike'."""
    data = CARS if vehicle_type.lower() == "car" else BIKES
    return ", ".join(item[0] for item in data)


In [ ]:
agent = create_agent(
    model=ChatOpenAI(model="gpt-4o-mini", temperature=0),
    tools=[find_car, find_bike, list_names],
    system_prompt="Use find_car for cars, find_bike for bikes, and list_names when the user "
                  "asks which names are available. Only use tool results. "
                  "If the match score is low, say you are not sure.",
)

# Type your clues, for example: show bike names, or: car huayra
while True:
    text = input("\nDescribe the vehicle (or 'quit'): ").strip()
    if text.lower() == "quit":
        break
    if text:
        result = agent.invoke({"messages": [{"role": "user", "content": text}]})
        for m in result["messages"]:
            for call in getattr(m, "tool_calls", []):
                print("Tool called:", call["name"])
        print("Answer:", result["messages"][-1].content)


Describe the vehicle (or 'quit'): find the max speed bike 
Tool called: list_names
Tool called: find_bike
Tool called: find_bike
Tool called: find_bike
Tool called: find_bike
Tool called: find_bike
Tool called: find_bike
Tool called: find_bike
Answer: The bike with the maximum speed is the **Kawasaki Ninja H2R**, which has a top speed of **400 km/h** and a power of **326 hp**.


KeyboardInterrupt: Interrupted by user